In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── 1. Load / prepare billings ──────────────────────────────────────────
df_billings = spark.table("post_renewal_churn.raw.billings") \
    .filter(F.col("prospect_outcome") != "Open") \
    .withColumn("datediff", F.datediff("closed_date", "prospect_renewal_date")) \
    .filter(F.col("datediff") < 29) \
    .withColumn("index", F.monotonically_increasing_id())


# Select all columns from raw_renewal_calls
df_calls = spark.table("post_renewal_churn.raw.renewal_calls") \
    .filter(F.col("analysed_call") == "1") \
    .filter(F.col("customer_renewal_response_category") != "null") \
    .filter(F.col("customer_renewal_response_category") != "Not Mentioned")

# ── 3. Join on co_ref — explicit condition to avoid ambiguous reference ──
df_joined = df_billings.join(
    df_calls,
    on=df_billings["co_ref"] == df_calls["co_ref"],
    how="left"
).filter(
    (F.col("call_date") >= F.col("prospect_renewal_date")) &
    (F.col("call_date") <= F.col("closed_date"))
).drop(df_calls["co_ref"])

# ── 4. Keep only the LATEST call per billing row (index) ────────────────
window = Window.partitionBy("index").orderBy(F.desc("call_date"))

df_latest_call = df_joined \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

df_latest_call.display()
df_final = df_latest_call
df_final.write.mode("overwrite").saveAsTable("post_renewal_churn.raw.joined_two_tables")

In [0]:
df_final.count()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import BooleanType

# ── STEP 0: Break lineage ────────────────────────────────────
df = spark.table("post_renewal_churn.raw.joined_two_tables")

# ── STEP 0b: Clean percentage-formatted numeric columns ─────
percentage_cols = ["#_of_Connection", "Discount_Amount"]
for col_name in percentage_cols:
    if col_name in df.columns:
        df = df.withColumn(
            col_name,
            F.regexp_replace(F.col(col_name), r"%", "").cast("double")
        )

# ── STEP 0c: Normalise Yes/No flag columns → INT ─────────────
flag_cols = [
    "Serious_Complaint", "Other_Complaint",
    "Discussion_on_Price_Increase", "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested", "Call_Reschedule_Request",
    "Agent_Flagged_Membership_Status_Alert", "Agent_Renewal_Initiation",
    "Explicit_Competitor_Mention", "Explicit_Switching_Intent",
    "Price_Switching_Mentioned", "Competitor_Value_Comparison",
    "Percentage_Price_Increase_Mentioned", "Monetary_Price_Increase_Mentioned",
    "Price_Range_Mentioned", "Customer_Asked_For_Justification",
    "Desire_To_Cancel", "Discount_Offered", "Analysed_Call",
]
for col_name in flag_cols:
    if col_name in df.columns:
        df = df.withColumn(
            col_name,
            F.when(F.upper(F.col(col_name)).isin("YES", "TRUE", "1"), F.lit(1))
             .when(F.upper(F.col(col_name)).isin("NO", "FALSE", "0"), F.lit(0))
             .otherwise(F.lit(None).cast("int"))
        )

# ── STEP 0d: Cast BOOLEAN columns → STRING ───────────────────
bool_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, BooleanType)]
for col_name in bool_cols:
    df = df.withColumn(col_name, F.col(col_name).cast("string"))

# ── STEP 0e: Cast string-fill columns → STRING ───────────────
# Ensures coalesce(col, lit("Unknown")) never tries to cast "Unknown" to BIGINT
string_fill_cols = [
    "Proforma_Auto_Renewal", "Proforma_World_Pay_Token",
    "Proforma_Account_Stage", "Proforma_Audit_Status",
    "Proforma_Membership_Status", "Proforma_Approved_Lists",
    "Current_Anchor_List", "Anchor_Group", "Connection_Group",
    "Mentioned_Competitors", "Competitor_Benefits_Mentioned",
    "Last_Band", "Payment_Timeframe", "Call_Direction",
    "Churn_Category", "Complaint_Category",
    "Customer_Reaction_Category", "Agent_Renewal_Pitch_Category",
    "Customer_Renewal_Response_Category", "Agent_Response_Category",
    "Membership_Renewal_Decision", "Topic_Introduced_By",
    "Customer_Response", "Justification_Category",
    "Reason_For_Renewal_Category", "Agent_Response_To_Cancel_Category",
    "Argument_That_Convinced_Customer_to_Stay_Category",
]
for col_name in string_fill_cols:
    if col_name in df.columns:
        df = df.withColumn(col_name, F.col(col_name).cast("string"))

# ── STEP 1: All medians in one groupBy per key ──────────────
conn_median = (
    df.groupBy("Connection_Group")
      .agg(
          F.percentile_approx("Connection_Net",          0.5).alias("_med_Connection_Net"),
          F.percentile_approx("Connection_Qty",          0.5).alias("_med_Connection_Qty"),
          F.percentile_approx("Starting_Connection_Net", 0.5).alias("_med_Starting_Connection_Net"),
          F.percentile_approx("Starting_Connection_Qty", 0.5).alias("_med_Starting_Connection_Qty"),
          F.percentile_approx("#_of_Connection",         0.5).alias("_med_of_Connection"),
          F.percentile_approx("Last_Connections",        0.5).alias("_med_Last_Connections"),
      )
)

yr_band_median = (
    df.groupBy("Renewal_Year", "Band")
      .agg(
          F.percentile_approx("Last_Years_Price",    0.5).alias("_med_Last_Years_Price"),
          F.percentile_approx("Total_Net_Paid",      0.5).alias("_med_Total_Net_Paid"),
          F.percentile_approx("Last_Total_Net_Paid", 0.5).alias("_med_Last_Total_Net_Paid"),
      )
)

band_median = (
    df.groupBy("Band")
      .agg(
          F.percentile_approx("Tenure_Years", 0.5).alias("_med_Tenure_Years"),
      )
)

global_medians = df.select(
    F.percentile_approx("Connection_Net",          0.5).alias("Connection_Net"),
    F.percentile_approx("Connection_Qty",          0.5).alias("Connection_Qty"),
    F.percentile_approx("Starting_Connection_Net", 0.5).alias("Starting_Connection_Net"),
    F.percentile_approx("Starting_Connection_Qty", 0.5).alias("Starting_Connection_Qty"),
    F.percentile_approx("#_of_Connection",         0.5).alias("#_of_Connection"),
    F.percentile_approx("Last_Connections",        0.5).alias("Last_Connections"),
    F.percentile_approx("Last_Years_Price",        0.5).alias("Last_Years_Price"),
    F.percentile_approx("Total_Net_Paid",          0.5).alias("Total_Net_Paid"),
    F.percentile_approx("Last_Total_Net_Paid",     0.5).alias("Last_Total_Net_Paid"),
    F.percentile_approx("Tenure_Years",            0.5).alias("Tenure_Years"),
).collect()[0]

# ── STEP 2: Mode for Payment_Timeframe ──────────────────────
mode_payment = (
    df.dropna(subset=["Payment_Timeframe"])
      .groupBy("Band", "Payment_Timeframe")
      .count()
      .withColumn(
          "rn",
          F.row_number().over(
              Window.partitionBy("Band").orderBy(F.col("count").desc())
          )
      )
      .filter(F.col("rn") == 1)
      .select("Band", F.col("Payment_Timeframe").alias("_mode_Payment_Timeframe"))
)

# ── STEP 3: All joins chained ────────────────────────────────
df = (
    df
    .join(F.broadcast(conn_median),    on="Connection_Group",       how="left")
    .join(F.broadcast(yr_band_median), on=["Renewal_Year", "Band"], how="left")
    .join(F.broadcast(band_median),    on="Band",                   how="left")
    .join(F.broadcast(mode_payment),   on="Band",                   how="left")
)

# ── STEP 3b: Pre-parse Last_Renewal to DATE ──────────────────
df = df.withColumn(
    "_Last_Renewal_Parsed",
    F.coalesce(
        F.try_to_date(F.col("Last_Renewal"), "yyyy-MM-dd"),
        F.try_to_date(F.col("Last_Renewal"), "dd-MM-yyyy"),
        F.try_to_date(F.col("Last_Renewal"), "MM-dd-yyyy"),
    )
)

# ── STEP 4: All fills in one select() ───────────────────────
def _med(col_name, lookup_col, global_val):
    return F.when(
        F.col(col_name).isNull(),
        F.coalesce(F.col(lookup_col), F.lit(global_val))
    ).otherwise(F.col(col_name)).alias(col_name)

def _coalesce_lit(col_name, value):
    return F.coalesce(F.col(col_name), F.lit(value)).alias(col_name)

lookup_drop_cols = {
    "_med_Connection_Net", "_med_Connection_Qty",
    "_med_Starting_Connection_Net", "_med_Starting_Connection_Qty",
    "_med_of_Connection", "_med_Last_Connections",
    "_med_Last_Years_Price", "_med_Total_Net_Paid", "_med_Last_Total_Net_Paid",
    "_med_Tenure_Years", "_mode_Payment_Timeframe",
    "_Last_Renewal_Parsed",
}

transform_cols = {
    "Connection_Net", "Connection_Qty",
    "Starting_Connection_Net", "Starting_Connection_Qty",
    "#_of_Connection", "Last_Connections",
    "Last_Years_Price", "Total_Net_Paid", "Last_Total_Net_Paid",
    "Tenure_Years", "Discount_Amount", "Last_Years_Date_Paid",
    "Proforma_Auto_Renewal", "Proforma_World_Pay_Token",
    "Proforma_Account_Stage", "Proforma_Audit_Status",
    "Proforma_Membership_Status", "Proforma_Approved_Lists",
    "Current_Anchor_List", "Payment_Timeframe",
    "Last_Renewal", "Last_Band", "Anchor_Group", "Connection_Group",
    "Mentioned_Competitors", "Competitor_Benefits_Mentioned",
    "Serious_Complaint", "Other_Complaint",
    "Discussion_on_Price_Increase", "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested", "Call_Reschedule_Request",
    "Agent_Flagged_Membership_Status_Alert", "Agent_Renewal_Initiation",
    "Explicit_Competitor_Mention", "Explicit_Switching_Intent",
    "Price_Switching_Mentioned", "Competitor_Value_Comparison",
    "Percentage_Price_Increase_Mentioned", "Monetary_Price_Increase_Mentioned",
    "Price_Range_Mentioned", "Customer_Asked_For_Justification",
    "Desire_To_Cancel", "Discount_Offered", "Analysed_Call",
    "Churn_Category", "Complaint_Category",
    "Customer_Reaction_Category", "Agent_Renewal_Pitch_Category",
    "Customer_Renewal_Response_Category", "Agent_Response_Category",
    "Membership_Renewal_Decision", "Topic_Introduced_By",
    "Customer_Response", "Call_Direction",
    "Justification_Category", "Reason_For_Renewal_Category",
    "Agent_Response_To_Cancel_Category",
    "Argument_That_Convinced_Customer_to_Stay_Category",
}

passthrough = [
    F.col(c) for c in df.columns
    if c not in lookup_drop_cols and c not in transform_cols
]

df = df.select(
    *passthrough,
    _med("Connection_Net",          "_med_Connection_Net",          global_medians["Connection_Net"]),
    _med("Connection_Qty",          "_med_Connection_Qty",          global_medians["Connection_Qty"]),
    _med("Starting_Connection_Net", "_med_Starting_Connection_Net", global_medians["Starting_Connection_Net"]),
    _med("Starting_Connection_Qty", "_med_Starting_Connection_Qty", global_medians["Starting_Connection_Qty"]),
    _med("#_of_Connection",         "_med_of_Connection",           global_medians["#_of_Connection"]),
    _med("Last_Connections",        "_med_Last_Connections",        global_medians["Last_Connections"]),
    _med("Last_Years_Price",    "_med_Last_Years_Price",    global_medians["Last_Years_Price"]),
    _med("Total_Net_Paid",      "_med_Total_Net_Paid",      global_medians["Total_Net_Paid"]),
    _med("Last_Total_Net_Paid", "_med_Last_Total_Net_Paid", global_medians["Last_Total_Net_Paid"]),
    _med("Tenure_Years",        "_med_Tenure_Years",        global_medians["Tenure_Years"]),
    _coalesce_lit("Discount_Amount",               0),
    _coalesce_lit("Current_Anchor_List",           "None"),
    _coalesce_lit("Anchor_Group",                  "Unknown"),
    _coalesce_lit("Connection_Group",              "Unknown"),
    _coalesce_lit("Proforma_Auto_Renewal",         "Unknown"),
    _coalesce_lit("Proforma_World_Pay_Token",      "Unknown"),
    _coalesce_lit("Proforma_Account_Stage",        "Unknown"),
    _coalesce_lit("Proforma_Audit_Status",         "Unknown"),
    _coalesce_lit("Proforma_Membership_Status",    "Unknown"),
    _coalesce_lit("Proforma_Approved_Lists",       "Unknown"),
    _coalesce_lit("Mentioned_Competitors",         "None"),
    _coalesce_lit("Competitor_Benefits_Mentioned", "None"),
    _coalesce_lit("Serious_Complaint",                              0),
    _coalesce_lit("Other_Complaint",                                0),
    _coalesce_lit("Discussion_on_Price_Increase",                   0),
    _coalesce_lit("Renewal_Impact_Due_to_Price_Increase",           0),
    _coalesce_lit("Discount_or_Waiver_Requested",                   0),
    _coalesce_lit("Call_Reschedule_Request",                        0),
    _coalesce_lit("Agent_Flagged_Membership_Status_Alert",          0),
    _coalesce_lit("Agent_Renewal_Initiation",                       0),
    _coalesce_lit("Explicit_Competitor_Mention",                    0),
    _coalesce_lit("Explicit_Switching_Intent",                      0),
    _coalesce_lit("Price_Switching_Mentioned",                      0),
    _coalesce_lit("Competitor_Value_Comparison",                    0),
    _coalesce_lit("Percentage_Price_Increase_Mentioned",            0),
    _coalesce_lit("Monetary_Price_Increase_Mentioned",              0),
    _coalesce_lit("Price_Range_Mentioned",                          0),
    _coalesce_lit("Customer_Asked_For_Justification",               0),
    _coalesce_lit("Desire_To_Cancel",                               0),
    _coalesce_lit("Discount_Offered",                               0),
    _coalesce_lit("Analysed_Call",                                  0),
    _coalesce_lit("Churn_Category",                     "Not recorded"),
    _coalesce_lit("Complaint_Category",                 "Not recorded"),
    _coalesce_lit("Customer_Reaction_Category",         "Not recorded"),
    _coalesce_lit("Agent_Renewal_Pitch_Category",       "Not recorded"),
    _coalesce_lit("Customer_Renewal_Response_Category", "Not recorded"),
    _coalesce_lit("Agent_Response_Category",            "Not recorded"),
    _coalesce_lit("Membership_Renewal_Decision",        "Not recorded"),
    _coalesce_lit("Topic_Introduced_By",                "Not recorded"),
    _coalesce_lit("Customer_Response",                  "Not recorded"),
    _coalesce_lit("Call_Direction",                     "Not recorded"),
    _coalesce_lit("Justification_Category",                             "Not applicable"),
    _coalesce_lit("Reason_For_Renewal_Category",                        "Not applicable"),
    _coalesce_lit("Agent_Response_To_Cancel_Category",                  "Not applicable"),
    _coalesce_lit("Argument_That_Convinced_Customer_to_Stay_Category",  "Not applicable"),
    F.when(
        F.col("Last_Years_Date_Paid").isNull() & F.col("_Last_Renewal_Parsed").isNotNull(),
        F.date_sub(F.col("_Last_Renewal_Parsed"), 365)
    ).otherwise(F.col("Last_Years_Date_Paid")).alias("Last_Years_Date_Paid"),
    F.when(
        F.col("Last_Renewal").isNull() & F.col("Renewal_Year").isNotNull(),
        F.make_date(F.col("Renewal_Year"), F.lit(1), F.lit(1))
    ).otherwise(F.col("_Last_Renewal_Parsed")).alias("Last_Renewal"),
    F.coalesce(F.col("Last_Band"), F.col("Band")).alias("Last_Band"),
    F.coalesce(F.col("Payment_Timeframe"), F.col("_mode_Payment_Timeframe")).alias("Payment_Timeframe"),
    F.when(F.col("Proforma_Date").isNull(),              1).otherwise(0).alias("Proforma_Date_Is_Null"),
    F.when(F.col("Registration_Date").isNull(),          1).otherwise(0).alias("Registration_Date_Is_Null"),
    F.when(F.col("Current_World_Pay_Token").isNotNull(), 1).otherwise(0).alias("Has_World_Pay_Token"),
)

# ── STEP 5: Tenure_Years secondary fill ─────────────────────
df = df.withColumn(
    "Tenure_Years",
    F.when(
        F.col("Tenure_Years").isNull() & F.col("Registration_Date").isNotNull(),
        F.round(F.datediff(F.current_date(), F.col("Registration_Date")) / 365.25, 1)
    ).otherwise(F.col("Tenure_Years").cast("INT"))
)

# ── STEP 6: Write result ─────────────────────────────────────
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("post_renewal_churn.raw.joined_two_tables_removed_nulls")

# ── VERIFY ───────────────────────────────────────────────────
print("=== Null check after imputation ===")
display(df)

In [0]:
df.count()


In [0]:
# pint all the columns null values count

df_nulls = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
display(df_nulls)
#

# Feature selection

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu
import pandas as pd
import numpy as np

spark = SparkSession.builder.getOrCreate()

# ── Load ──────────────────────────────────────────────────────────────────────
df_spark = spark.table("post_renewal_churn.raw.joined_two_tables_removed_nulls")
df = df_spark.toPandas()

target        = "Prospect_Outcome"
target_binary = (df[target] == "Churned").astype(int)

# ─────────────────────────────────────────────────────────────────────────────
# HYPOTHESIS FRAMEWORK
# ─────────────────────────────────────────────────────────────────────────────
#
# CHI-SQUARE TEST (Categorical features)
# ───────────────────────────────────────
# H0: The distribution of [feature] is INDEPENDENT of prospect_outcome
#     (i.e. the feature has NO association with churn vs won)
# H1: The distribution of [feature] is DEPENDENT on prospect_outcome
#     (i.e. the feature IS associated with churn vs won)
# Decision: Reject H0 if p < bonferroni_alpha → feature is significant
# Effect size: Cramér's V (0=no association, 1=perfect association)
#
# T-TEST / MANN-WHITNEY U (Continuous features)
# ───────────────────────────────────────────────
# H0: mean([feature] | Won) == mean([feature] | Churned)
#     (i.e. the feature does NOT differ between churned and won customers)
# H1: mean([feature] | Won) != mean([feature] | Churned)
#     (i.e. the feature DOES differ between churned and won customers)
# Decision: Reject H0 if p < bonferroni_alpha → feature is significant
# Effect size: Cohen's d (0.2=small, 0.5=medium, 0.8=large)
#
# Multiple testing correction: Bonferroni (α = 0.05 / number of tests)
# This controls the family-wise error rate — reduces false positives
# from running many simultaneous hypothesis tests.
# ─────────────────────────────────────────────────────────────────────────────

DROP_COLS = {
    "co_ref", "prospect_outcome", "renewal_month", "date_time_out",
    "proforma_date", "registration_date", "prospect_renewal_date",
    "closed_date", "call_date", "last_years_date_paid", "last_renewal",
    "current_anchor_list", "competitor_benefits_mentioned",
    "proforma_approved_lists", "call_id", "index",
    "price_range_mentioned", "desire_to_cancel", "analysed_call",
    "c20", "competitor_value_comparison", "has_world_pay_token",
}

CHI_SQ_COLS = [
    "current_auto_renewal_flag", "current_world_pay_token",
    "proforma_auto_renewal", "proforma_world_pay_token",
    "mentioned_competitors", "membership_renewal_decision",
    "band", "last_band", "renewal_year", "payment_method",
    "prospect_status", "proforma_account_stage", "proforma_audit_status",
    "proforma_membership_status", "churn_category", "complaint_category",
    "customer_reaction_category", "agent_renewal_pitch_category",
    "customer_renewal_response_category", "agent_response_category",
    "topic_introduced_by", "customer_response", "call_direction",
    "justification_category", "reason_for_renewal_category",
    "agent_response_to_cancel_category",
    "argument_that_convinced_customer_to_stay_category",
    "serious_complaint", "other_complaint", "discussion_on_price_increase",
    "renewal_impact_due_to_price_increase", "discount_or_waiver_requested",
    "call_reschedule_request", "agent_flagged_membership_status_alert",
    "agent_renewal_initiation", "explicit_competitor_mention",
    "explicit_switching_intent", "price_switching_mentioned",
    "percentage_price_increase_mentioned", "monetary_price_increase_mentioned",
    "customer_asked_for_justification", "discount_offered",
    "proforma_date_is_null", "registration_date_is_null",
    "call_year", "call_number",
]

T_TEST_COLS = [
    "sustainability_score", "total_renewal_score_new", "auto_renewal_score",
    "status_scores", "anchoring_score", "tenure_scores", "current_anchorings",
    "renewal_score_at_release", "starting_net", "starting_vat",
    "starting_gross", "starting_membership_net", "starting_package_net",
    "starting_pqq_net", "gross", "membership_net", "package_net", "pqq_net",
    "amount", "total_amount", "datediff", "connection_net", "connection_qty",
    "starting_connection_net", "starting_connection_qty", "of_connection",
    "last_connections", "last_years_price", "total_net_paid",
    "last_total_net_paid", "tenure_years", "discount_amount", "payment_timeframe",
]

# ─────────────────────────────────────────────────────────────────────────────
# ENCODING
# ─────────────────────────────────────────────────────────────────────────────

def encode_binary_str(series):
    mapping = {"y": 1, "n": 0, "true": 1, "false": 0,
               "yes": 1, "no": 0, "unknown": np.nan}
    return series.str.lower().map(mapping)

def standardise_call_direction(series):
    return series.str.upper().str.replace("_", "", regex=False)\
                 .map({"OUTBOUND": "Outbound", "INBOUND": "Inbound"})

df["current_auto_renewal_flag"]    = encode_binary_str(df["current_auto_renewal_flag"])
df["current_world_pay_token"]      = encode_binary_str(df["current_world_pay_token"])
df["proforma_auto_renewal"]        = encode_binary_str(df["proforma_auto_renewal"])
df["proforma_world_pay_token"]     = encode_binary_str(df["proforma_world_pay_token"])
df["mentioned_competitors"]        = encode_binary_str(df["mentioned_competitors"])
df["membership_renewal_decision"]  = encode_binary_str(df["membership_renewal_decision"])
df["call_direction"]               = standardise_call_direction(df["call_direction"])
df["tenure_group"]                 = df["tenure_group"].replace("4+", "4").astype(float)

T_TEST_COLS.append("tenure_group")
if "tenure_group" in CHI_SQ_COLS:
    CHI_SQ_COLS.remove("tenure_group")

ordinal_map = {"1": 1, "2": 2, "3": 3, "4 to 9": 6.5, "10+": 10}
df["anchor_group"]     = df["anchor_group"].map(ordinal_map)
df["connection_group"] = df["connection_group"].map(ordinal_map)
T_TEST_COLS += ["anchor_group", "connection_group"]
for c in ["anchor_group", "connection_group"]:
    if c in CHI_SQ_COLS:
        CHI_SQ_COLS.remove(c)

# ─────────────────────────────────────────────────────────────────────────────
# BONFERRONI CORRECTION
# ─────────────────────────────────────────────────────────────────────────────
total_tests      = len(CHI_SQ_COLS) + len(T_TEST_COLS)
bonferroni_alpha = 0.05 / total_tests

print("=" * 65)
print("HYPOTHESIS TESTING FOR CHURN PREDICTION")
print("=" * 65)
print(f"\nTarget variable  : {target}  (Won=0, Churned=1)")
print(f"Total tests      : {total_tests}  ({len(CHI_SQ_COLS)} chi-square + {len(T_TEST_COLS)} t-test)")
print(f"Raw α            : 0.05")
print(f"Bonferroni α     : 0.05 / {total_tests} = {bonferroni_alpha:.6f}")
print(f"\nCHI-SQUARE HYPOTHESES (per categorical feature):")
print(f"  H0: [feature] is independent of {target} — no association")
print(f"  H1: [feature] is dependent on {target}  — significant association")
print(f"\nT-TEST HYPOTHESES (per continuous feature):")
print(f"  H0: mean([feature] | Won) = mean([feature] | Churned)")
print(f"  H1: mean([feature] | Won) ≠ mean([feature] | Churned)")
print(f"\nDecision rule: Reject H0 if p_value < {bonferroni_alpha:.6f}")

won_mask   = df[target] == "Won"
churn_mask = df[target] == "Churned"

# ─────────────────────────────────────────────────────────────────────────────
# 1. CHI-SQUARE TEST
# ─────────────────────────────────────────────────────────────────────────────
chi_results = []

for col in CHI_SQ_COLS:
    if col not in df.columns:
        continue
    try:
        temp = df[[col, target]].dropna()
        ct   = pd.crosstab(temp[col], temp[target])
        if ct.shape[0] < 2 or ct.shape[1] < 2:
            raise ValueError("Degenerate contingency table — only one category present")

        chi2, p, dof, expected = chi2_contingency(ct)

        # Cramér's V effect size
        n = ct.values.sum()
        v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

        # Effect size interpretation
        if   v >= 0.5: effect_label = "Very Strong"
        elif v >= 0.3: effect_label = "Strong"
        elif v >= 0.1: effect_label = "Moderate"
        else:          effect_label = "Weak"

        reject_h0 = p < bonferroni_alpha

        chi_results.append({
            "Feature"         : col,
            "H0"              : f"'{col}' is independent of {target}",
            "H1"              : f"'{col}' is associated with {target}",
            "Chi2_Statistic"  : round(chi2, 4),
            "Degrees_Freedom" : dof,
            "P_Value"         : round(p, 8),
            "CramersV"        : round(v, 4),
            "Effect_Strength" : effect_label,
            "Reject_H0"       : "YES — significant" if reject_h0 else "NO  — not significant",
            "Conclusion"      : (f"'{col}' IS associated with churn (use in model)"
                                 if reject_h0 else
                                 f"'{col}' is NOT associated with churn (consider dropping)"),
        })
    except Exception as e:
        chi_results.append({
            "Feature": col, "H0": None, "H1": None,
            "Chi2_Statistic": None, "Degrees_Freedom": None,
            "P_Value": None, "CramersV": None,
            "Effect_Strength": None,
            "Reject_H0": f"ERROR", "Conclusion": str(e),
        })

# ─────────────────────────────────────────────────────────────────────────────
# 2. T-TEST + MANN-WHITNEY U
# ─────────────────────────────────────────────────────────────────────────────
t_results = []

for col in T_TEST_COLS:
    if col not in df.columns:
        continue
    try:
        won_vals   = df.loc[won_mask,   col].dropna()
        churn_vals = df.loc[churn_mask, col].dropna()

        if len(won_vals) < 2 or len(churn_vals) < 2:
            raise ValueError("Insufficient data in one group")

        # Welch's t-test (does not assume equal variance)
        t_stat, t_p = ttest_ind(won_vals, churn_vals, equal_var=False)

        # Mann-Whitney U (non-parametric — no normality assumption)
        u_stat, u_p = mannwhitneyu(won_vals, churn_vals, alternative="two-sided")

        # Cohen's d
        pooled_std = np.sqrt((won_vals.std()**2 + churn_vals.std()**2) / 2)
        cohens_d   = abs(won_vals.mean() - churn_vals.mean()) / pooled_std if pooled_std > 0 else 0

        # Effect size interpretation
        if   cohens_d >= 0.8: effect_label = "Large"
        elif cohens_d >= 0.5: effect_label = "Medium"
        elif cohens_d >= 0.2: effect_label = "Small"
        else:                 effect_label = "Negligible"

        # Direction of effect
        direction = "higher in Churned" if churn_vals.mean() > won_vals.mean() else "higher in Won"

        reject_h0_t = t_p < bonferroni_alpha
        reject_h0_u = u_p < bonferroni_alpha
        reject_h0   = reject_h0_t or reject_h0_u

        t_results.append({
            "Feature"           : col,
            "H0"                : f"mean({col} | Won) = mean({col} | Churned)",
            "H1"                : f"mean({col} | Won) ≠ mean({col} | Churned)",
            "Won_Mean"          : round(float(won_vals.mean()),   4),
            "Churned_Mean"      : round(float(churn_vals.mean()), 4),
            "Direction"         : direction,
            "T_Statistic"       : round(float(t_stat), 4),
            "T_P_Value"         : round(float(t_p),    8),
            "U_Statistic"       : round(float(u_stat), 4),
            "U_P_Value"         : round(float(u_p),    8),
            "Cohens_d"          : round(float(cohens_d), 4),
            "Effect_Strength"   : effect_label,
            "Reject_H0 (t)"     : "YES" if reject_h0_t else "NO",
            "Reject_H0 (U)"     : "YES" if reject_h0_u else "NO",
            "Conclusion"        : (f"'{col}' differs significantly — {direction} (use in model)"
                                   if reject_h0 else
                                   f"'{col}' does NOT differ between groups (consider dropping)"),
        })
    except Exception as e:
        t_results.append({
            "Feature": col, "H0": None, "H1": None,
            "Won_Mean": None, "Churned_Mean": None, "Direction": None,
            "T_Statistic": None, "T_P_Value": None,
            "U_Statistic": None, "U_P_Value": None,
            "Cohens_d": None, "Effect_Strength": None,
            "Reject_H0 (t)": "ERROR", "Reject_H0 (U)": "ERROR",
            "Conclusion": str(e),
        })

# ─────────────────────────────────────────────────────────────────────────────
# DISPLAY RESULTS
# ─────────────────────────────────────────────────────────────────────────────
chi_df = pd.DataFrame(chi_results).sort_values("CramersV", ascending=False)
t_df   = pd.DataFrame(t_results).sort_values("Cohens_d",  ascending=False)

def to_spark_safe(pdf):
    return spark.createDataFrame(pdf.astype(str).fillna("N/A"))

print(f"\n{'='*65}")
print("CHI-SQUARE RESULTS  (sorted by Cramér's V — effect size)")
print(f"{'='*65}")
display(to_spark_safe(chi_df))

print(f"\n{'='*65}")
print("T-TEST + MANN-WHITNEY U RESULTS  (sorted by Cohen's d)")
print(f"{'='*65}")
display(to_spark_safe(t_df))

# ── Summary ───────────────────────────────────────────────────────────────────
chi_sig  = chi_df[chi_df["Reject_H0"].str.startswith("YES")]
t_sig    = t_df[t_df["Reject_H0 (t)"].eq("YES") | t_df["Reject_H0 (U)"].eq("YES")]
chi_drop = chi_df[chi_df["Reject_H0"].str.startswith("NO")]
t_drop   = t_df[t_df["Reject_H0 (t)"].eq("NO")  & t_df["Reject_H0 (U)"].eq("NO")]

print(f"\n{'='*65}")
print("SUMMARY")
print(f"{'='*65}")
print(f"\n  H0 REJECTED  (keep for model) : {len(chi_sig)} categorical + {len(t_sig)} continuous")
print(f"  H0 RETAINED  (consider drop)  : {len(chi_drop)} categorical + {len(t_drop)} continuous")

print(f"\n  Significant categorical features (H0 rejected):")
for _, row in chi_sig[["Feature","CramersV","Effect_Strength"]].iterrows():
    print(f"    ✓  {row['Feature']:<55} V={row['CramersV']}  [{row['Effect_Strength']}]")

print(f"\n  Significant continuous features (H0 rejected):")
for _, row in t_sig[["Feature","Won_Mean","Churned_Mean","Cohens_d","Effect_Strength","Direction"]].iterrows():
    print(f"    ✓  {row['Feature']:<40} d={row['Cohens_d']}  [{row['Effect_Strength']}]  {row['Direction']}")

print(f"\n  Features where H0 was NOT rejected (consider dropping):")
all_drop = list(chi_drop["Feature"]) + list(t_drop["Feature"])
print(f"    {', '.join(all_drop)}")